<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/cursos/machine-learning-python2/Clase06/cuaderno-05b-regresion-times-listed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Una alternativa para la regresión: predecir `Times Listed`

En el cuaderno anterior intentamos predecir **`Rating`** mediante una regresión lineal y obtuvimos un **R² ≈ 0.2375**. Esto indicaba que las variables elegidas explicaban solo una parte limitada de la variabilidad de la valoración de los videojuegos.

En este cuaderno breve vamos a plantear una pregunta diferente:

> **¿Podemos obtener un modelo lineal más explicativo si intentamos predecir `Times Listed`?**

La idea surge al observar la matriz de correlación del cuaderno anterior. Varias variables relacionadas con la actividad de los usuarios —como `Plays`, `Playing`, `Backlogs` y `Wishlist`— parecen estar más directamente vinculadas con la cantidad de veces que un juego aparece en listas que con su valoración.

Usaremos el mismo dataset **Popular Video Games 1980-2023** y mantendremos el mismo criterio de división entre entrenamiento y prueba para que la comparación sea lo más directa posible.

## Un hallazgo importante antes de entrenar

Al preparar el dataset aparece una particularidad: las columnas **`Times Listed`** y **`Number of Reviews`** contienen los mismos valores.

Esto significa que no sería válido utilizar `Number of Reviews` como variable predictora para estimar `Times Listed`, porque el modelo estaría recibiendo prácticamente la respuesta entre sus datos de entrada.

Por ese motivo, en este experimento:

- `Times Listed` será la **variable objetivo**;
- `Number of Reviews` será **excluida**;
- utilizaremos como predictoras `Rating`, `Plays`, `Playing`, `Backlogs` y `Wishlist`.

Primero cargamos y preparamos los datos, y verificamos si las dos columnas son efectivamente idénticas.

In [ ]:
# ==========================================================
# CARGA, PREPARACIÓN Y VERIFICACIÓN DEL DATASET
# ==========================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Cargamos el mismo dataset utilizado en el cuaderno anterior.
ruta_local = "/content/games.csv"

if os.path.exists(ruta_local):
    ruta_csv = ruta_local
else:
    import kagglehub
    ruta_dataset = kagglehub.dataset_download(
        "arnabchaki/popular-video-games-1980-2023"
    )
    ruta_csv = os.path.join(ruta_dataset, "games.csv")

games = pd.read_csv(ruta_csv)

# Convertimos valores como "3.9K" o "17K" a números.
def convertir_k_a_numero(valor):
    if pd.isna(valor):
        return np.nan

    valor = str(valor).strip().upper()

    if valor.endswith("K"):
        return float(valor[:-1]) * 1000

    return float(valor)

columnas_convertir = [
    "Times Listed",
    "Number of Reviews",
    "Plays",
    "Playing",
    "Backlogs",
    "Wishlist"
]

games_transformado = games.copy()

for columna in columnas_convertir:
    games_transformado[columna] = (
        games_transformado[columna].apply(convertir_k_a_numero)
    )

# Verificamos el hallazgo observado en el cuaderno anterior.
son_identicas = (
    games_transformado["Times Listed"]
    == games_transformado["Number of Reviews"]
).all()

print("¿Times Listed y Number of Reviews son idénticas?:", son_identicas)
print("Cantidad de registros:", len(games_transformado))

## Entrenamiento y evaluación

Ahora entrenamos una regresión lineal con `Times Listed` como objetivo.

Para evitar la fuga de información, eliminamos `Number of Reviews`. También mantenemos `random_state=42`, igual que en el experimento anterior.

Además de calcular MAE, MSE, RMSE y R², compararemos directamente el nuevo **R²** con el valor aproximado **0.2375** obtenido al predecir `Rating`.

Recordemos que MAE y RMSE ahora estarán expresados en **cantidad de apariciones en listas**, por lo que sus valores no pueden compararse directamente con los errores medidos en puntos de `Rating`.

In [ ]:
# ==========================================================
# NUEVO MODELO: PREDICCIÓN DE TIMES LISTED
# ==========================================================

columnas_modelo = [
    "Rating",
    "Times Listed",
    "Plays",
    "Playing",
    "Backlogs",
    "Wishlist"
]

games_modelo = games_transformado[columnas_modelo].dropna().copy()

# Variables predictoras y objetivo
X = games_modelo[
    ["Rating", "Plays", "Playing", "Backlogs", "Wishlist"]
]
y = games_modelo["Times Listed"]

# División entrenamiento/prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Entrenamiento
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# Predicciones
y_pred = modelo.predict(X_test)

# Métricas
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 4))
print("MSE :", round(mse, 4))
print("RMSE:", round(rmse, 4))
print("R²  :", round(r2, 4))

# Comparación con el modelo anterior
r2_rating = 0.2375

print("\nComparación de R²:")
print("Modelo para Rating      :", r2_rating)
print("Modelo para Times Listed:", round(r2, 4))

print("\nConclusión:")
if r2 > r2_rating:
    mejora = (r2 - r2_rating) * 100
    print(
        "El nuevo modelo obtiene un R² mayor que el modelo utilizado para "
        "predecir Rating. La mejora absoluta es de "
        f"{mejora:.2f} puntos porcentuales de varianza explicada."
    )
    print(
        "Esto respalda la hipótesis de que las variables de actividad de los "
        "usuarios mantienen una relación lineal más útil con Times Listed."
    )
else:
    print(
        "El nuevo modelo no mejora el R² obtenido al predecir Rating. "
        "Por lo tanto, cambiar la variable objetivo no produjo la mejora esperada."
    )

## Qué debemos observar

El dato central de este experimento es **R²**, porque permite comparar qué proporción de la variabilidad de cada variable objetivo logra explicar su respectivo modelo.

Los errores MAE y RMSE siguen siendo útiles para evaluar el nuevo modelo, pero no deben compararse numéricamente con los del cuaderno anterior: `Rating` y `Times Listed` están expresados en escalas completamente diferentes.

La última parte de la celda anterior genera automáticamente una conclusión a partir del resultado obtenido.

---

### Autoría y atribución

Este cuaderno forma parte del material educativo desarrollado por **VintaBytes**.

📧 **Contacto:** [vintabytes@gmail.com](mailto:vintabytes@gmail.com)

🌐 **Repositorio oficial:** [github.com/VintaBytes/Ciencia-de-datos](https://github.com/VintaBytes/Ciencia-de-datos)

Se permite compartir y reutilizar este material respetando la autoría y manteniendo visible esta atribución.

---
